# 17. Usuarios y contraseñas

Prueba del registro de usuarios y de la verificación del ingreso (tarea 20 del backlog, HU-08). Las funciones se encuentran en `src/usuarios.py`. La prueba usa un archivo temporal, evitando modificar el archivo de usuarios del prototipo.

In [1]:
import sys
sys.path.append("../src")

In [2]:
import tempfile
from pathlib import Path
from usuarios import registrar_usuario, verificar_ingreso, cargar_usuarios

carpeta_prueba = tempfile.TemporaryDirectory()
ruta_prueba = Path(carpeta_prueba.name) / "USUARIOS_PRUEBA.json"

## 1. Registro y verificación

Las contraseñas no se guardan: se guarda una sal aleatoria por usuario y el hash de la contraseña con esa sal, calculado con PBKDF2 y SHA-256 en 600.000 iteraciones. Al ingresar, se vuelve a calcular el hash con la contraseña escrita y se compara con el guardado. Un usuario inexistente y una contraseña incorrecta devuelven el mismo resultado, para no revelar cuál de los dos datos falló.

In [3]:
registrar_usuario(ruta_prueba, "prueba", "clave1234", "OPERADOR")
print(verificar_ingreso(ruta_prueba, "prueba", "clave1234"))
print(verificar_ingreso(ruta_prueba, "prueba", "Clave12345"))
print(verificar_ingreso(ruta_prueba, "nadie", "clave1234"))
cargar_usuarios(ruta_prueba)

OPERADOR
None
None


{'prueba': {'rol': 'OPERADOR',
  'sal': 'd674227896c38be2c4296e6a73015d89',
  'hash': 'b2a317dd614ec71fbe02a4274037e16e60add8c7872b9f1b5503eac53ebfe830'}}

## 2. Validación del registro

Se intenta registrar un usuario repetido, con una contraseña corta y un rol inexistente. La validación informa los tres errores juntos y no registra nada.

In [4]:
try:
    registrar_usuario(ruta_prueba, "prueba", "corta", "ADMIN")
except ValueError as error:
    print("Error esperado:\n", error)

Error esperado:
 El usuario prueba ya existe
La contraseña debe tener al menos 8 caracteres
El rol debe ser uno de ['OPERADOR', 'RESPONSABLE']


## Conclusión

- El ingreso es correcto solo con el usuario y la contraseña exactos. La comparación distingue mayúsculas y minúsculas en la contraseña, no en el usuario.
- El archivo guarda la sal y el hash de cada usuario, nunca la contraseña.
- El registro rechaza usuarios repetidos, contraseñas de menos de 8 caracteres y roles distintos de Operador y Responsable.